# Vietnamese Embedding Model Evaluation for Semantic Similarity
## Dataset: SEACrowd/visim400

This notebook evaluates multiple Vietnamese embedding models on the ViSim-400 dataset,
a Vietnamese semantic similarity benchmark with 400 sentence pairs annotated with similarity scores.

**Metric**: Pearson & Spearman correlation between model cosine similarities and human scores.

## 1. Install Dependencies 

In [1]:
%pip install sentence-transformers datasets seacrowd scipy numpy pandas matplotlib seaborn pyvi -q


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Load the ViSim-400 Dataset

In [1]:
import pandas as pd
import seacrowd as sc

# List all datasets
dset_names = sc.list_datasets()

# Load the dataset using the default config
dataset = sc.load_dataset("visim400", schema="seacrowd")


# from datasets import load_dataset

# # Load dataset — requires 'seacrowd' to be installed
# # ViSim-400 has sentence pairs with human similarity scores
# dataset = load_dataset(
#     "SEACrowd/visim400",
#     trust_remote_code=True
# )

print("Dataset splits:", dataset)
print("\nFirst example:")
print(dataset[list(dataset.keys())[0]][0])

/Users/thanhdat/Workspace/Dat_all_Mac_projects/Demo/vi_embed_eva/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/402 [00:00<?, ?it/s]


TypeError: DownloadConfig.__init__() got an unexpected keyword argument 'trust_remote_code'

In [ ]:
# Inspect columns and build a unified DataFrame
split = list(dataset.keys())[0]   # use whatever split is available
df = dataset[split].to_pandas()
print("Columns:", df.columns.tolist())
print(f"Total pairs: {len(df)}")
df.head(5)

In [ ]:
# ---- Adapt column names if needed ----
# ViSim-400 typically has: 'text_1', 'text_2', 'label' (or 'score', 'similarity')
# Adjust the mapping below to match the actual column names shown above.

COL_SENT1  = "text_1"    # <-- change if your df uses a different name
COL_SENT2  = "text_2"    # <-- change if your df uses a different name
COL_SCORE  = "label"     # <-- change if your df uses 'score' / 'similarity'

sentences1    = df[COL_SENT1].tolist()
sentences2    = df[COL_SENT2].tolist()
human_scores  = df[COL_SCORE].astype(float).tolist()

print(f"Example pair:")
print(f"  S1: {sentences1[0]}")
print(f"  S2: {sentences2[0]}")
print(f"  Score: {human_scores[0]}")

## 3. Define Models to Evaluate

In [ ]:
MODELS = [
    {
        "name": "dangvantuan/vietnamese-embedding",
        "label": "VN-Embedding (PhoBERT)",
        "needs_pyvi": True,   # requires Vietnamese word tokenization
    },
    {
        "name": "AITeamVN/Vietnamese_Embedding",
        "label": "AITeamVN VN-Embed (BGE-M3)",
        "needs_pyvi": False,
    },
    {
        "name": "dangvantuan/vietnamese-document-embedding",
        "label": "VN-DocEmbed (GTE-multilingual)",
        "needs_pyvi": False,
        "trust_remote_code": True,
    },
    {
        "name": "keepitreal/vietnamese-sbert",
        "label": "VN-SBERT (baseline)",
        "needs_pyvi": False,
    },
    {
        "name": "BAAI/bge-m3",
        "label": "BGE-M3 (multilingual)",
        "needs_pyvi": False,
    },
]

print(f"Will evaluate {len(MODELS)} models.")

## 4. Evaluation Helper Functions

In [ ]:
import numpy as np
from scipy import stats
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

try:
    from pyvi.ViTokenizer import tokenize as vi_tokenize
except ImportError:
    vi_tokenize = None
    print("pyvi not available — models requiring it will be skipped.")


def compute_cosine_similarities(model, sents1, sents2, batch_size=32):
    """Encode two sentence lists and return cosine similarities."""
    emb1 = model.encode(sents1, batch_size=batch_size, show_progress_bar=True,
                         convert_to_tensor=True, normalize_embeddings=True)
    emb2 = model.encode(sents2, batch_size=batch_size, show_progress_bar=True,
                         convert_to_tensor=True, normalize_embeddings=True)
    sims = cos_sim(emb1, emb2).diagonal().cpu().numpy()
    return sims


def evaluate_model(model_cfg, sents1, sents2, gold_scores):
    """Load a model, compute similarities, return Pearson & Spearman r."""
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_cfg['label']}")
    print(f"{'='*60}")

    # Optionally apply PyVI tokenization
    if model_cfg.get("needs_pyvi"):
        if vi_tokenize is None:
            print("  Skipping — pyvi not installed.")
            return None
        s1 = [vi_tokenize(s) for s in sents1]
        s2 = [vi_tokenize(s) for s in sents2]
    else:
        s1, s2 = sents1, sents2

    # Load model
    trust = model_cfg.get("trust_remote_code", False)
    model = SentenceTransformer(model_cfg["name"],
                                trust_remote_code=trust)

    # Compute cosine similarities
    pred_sims = compute_cosine_similarities(model, s1, s2)

    # Normalize gold scores to [0, 1] if needed (ViSim-400 uses 0–4 scale)
    gold = np.array(gold_scores)
    if gold.max() > 1.0:
        gold = gold / gold.max()

    pearson_r,  pearson_p  = stats.pearsonr(gold, pred_sims)
    spearman_r, spearman_p = stats.spearmanr(gold, pred_sims)

    print(f"  Pearson  r = {pearson_r:.4f}  (p={pearson_p:.2e})")
    print(f"  Spearman r = {spearman_r:.4f}  (p={spearman_p:.2e})")

    return {
        "Model": model_cfg["label"],
        "Pearson r": round(pearson_r, 4),
        "Spearman r": round(spearman_r, 4),
        "pred_sims": pred_sims,
        "gold": gold,
    }

print("Helper functions defined.")

## 5. Run Evaluation

In [ ]:
results = []

for cfg in MODELS:
    try:
        result = evaluate_model(cfg, sentences1, sentences2, human_scores)
        if result is not None:
            results.append(result)
    except Exception as e:
        print(f"  ERROR for {cfg['label']}: {e}")

print("\nAll evaluations complete.")

## 6. Summary Table

In [ ]:
summary = pd.DataFrame([{"Model": r["Model"],
                          "Pearson r": r["Pearson r"],
                          "Spearman r": r["Spearman r"]} for r in results])

summary = summary.sort_values("Spearman r", ascending=False).reset_index(drop=True)
summary.index += 1  # 1-based rank
summary.index.name = "Rank"

print("\n=== Evaluation Results on ViSim-400 ===")
print(summary.to_string())
summary

## 7. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

# --- Bar chart: Pearson & Spearman side-by-side ---
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(summary))
width = 0.35
bars1 = ax.bar(x - width/2, summary["Pearson r"],  width, label="Pearson r",  color="#4C9BE8")
bars2 = ax.bar(x + width/2, summary["Spearman r"], width, label="Spearman r", color="#F4845F")

ax.set_xticks(x)
ax.set_xticklabels(summary["Model"], rotation=20, ha="right")
ax.set_ylabel("Correlation")
ax.set_title("Vietnamese Embedding Models — ViSim-400 Evaluation")
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()

In [ ]:
# --- Scatter plots: predicted vs gold similarity per model ---
n = len(results)
cols = min(3, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
axes = np.array(axes).flatten()

for i, r in enumerate(results):
    ax = axes[i]
    ax.scatter(r["gold"], r["pred_sims"], alpha=0.5, s=20, color="#4C9BE8")
    
    # Regression line
    m, b = np.polyfit(r["gold"], r["pred_sims"], 1)
    x_line = np.linspace(r["gold"].min(), r["gold"].max(), 100)
    ax.plot(x_line, m * x_line + b, color="#F4845F", linewidth=2)
    
    ax.set_xlabel("Human Score (normalized)")
    ax.set_ylabel("Predicted Cosine Similarity")
    ax.set_title(f"{r['Model']}\nPearson={r['Pearson r']:.3f} | Spearman={r['Spearman r']:.3f}")

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig("scatter_plots.png", dpi=150)
plt.show()

## 8. Error Analysis — Hard Cases

In [ ]:
# Identify pairs where the best model was most wrong
best = results[0]  # highest Spearman r
errors = np.abs(best["pred_sims"] - best["gold"])

error_df = pd.DataFrame({
    "sentence_1": sentences1,
    "sentence_2": sentences2,
    "gold": best["gold"],
    "predicted": best["pred_sims"],
    "abs_error": errors,
}).sort_values("abs_error", ascending=False)

print(f"Top 10 hardest cases for: {best['Model']}")
error_df.head(10)

## 9. Save Results to CSV

In [ ]:
summary.to_csv("visim400_results.csv")
error_df.to_csv("error_analysis.csv", index=False)
print("Results saved to visim400_results.csv and error_analysis.csv")

---
## Notes

- **ViSim-400** is a Vietnamese semantic similarity dataset with scores on a 0–4 scale. Scores are normalized to [0,1] before computing correlations.
- `dangvantuan/vietnamese-embedding` requires `pyvi` for Vietnamese word segmentation — the tokenization step is applied automatically above.
- `AITeamVN/Vietnamese_Embedding` (BGE-M3 fine-tuned) generally excels at retrieval tasks with its 2048-token context.
- `dangvantuan/vietnamese-document-embedding` is best for long documents (up to 8096 tokens).
- For GPU acceleration, ensure CUDA is available — `SentenceTransformer` uses it automatically.
- If you want to add more models, just append a new dict to the `MODELS` list in Section 3.